In [6]:
import pandas as pd

captions_df = pd.read_csv("../data/captions.txt")
print("Shape:", captions_df.shape)
print(captions_df.head(10))
print("\nColumn names:", captions_df.columns.tolist())
print("\nUnique images:", captions_df.iloc[:, 0].nunique())

Shape: (40455, 2)
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   
5  1001773457_577c3a7d70.jpg   
6  1001773457_577c3a7d70.jpg   
7  1001773457_577c3a7d70.jpg   
8  1001773457_577c3a7d70.jpg   
9  1001773457_577c3a7d70.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
5         A black dog and a spotted dog are fighting  
6  A black dog and a tri-colored dog playing with...  
7  A black dog and a white dog with brown spots a...  
8  Two dogs of different breeds looking at each o...  
9    Two dogs on pavement moving toward each other .  

Column names: ['image',

In [7]:
import sys
sys.path.append("../src")

from data_loader import load_captions, preprocess_captions, build_vocabulary, get_max_caption_length

captions_df = load_captions("../data/captions.txt")
captions_df = preprocess_captions(captions_df)

print(captions_df[["caption", "clean_caption"]].head(5))

word_to_idx, idx_to_word = build_vocabulary(captions_df["clean_caption"].tolist(), min_word_freq=5)
max_len = get_max_caption_length(captions_df["clean_caption"].tolist())

print(f"\nVocabulary size: {len(word_to_idx)}")
print(f"Max caption length: {max_len}")



                                             caption  \
0  A child in a pink dress is climbing up a set o...   
1              A girl going into a wooden building .   
2   A little girl climbing into a wooden playhouse .   
3  A little girl climbing the stairs to her playh...   
4  A little girl in a pink dress going into a woo...   

                                       clean_caption  
0  startseq a child in a pink dress is climbing u...  
1  startseq a girl going into a wooden building e...  
2  startseq a little girl climbing into a wooden ...  
3  startseq a little girl climbing the stairs to ...  
4  startseq a little girl in a pink dress going i...  

Vocabulary size: 2987
Max caption length: 37


In [9]:
import sys
sys.path.append("../src")

from encoder import build_encoder, extract_features
import os

encoder = build_encoder()
print("Encoder loaded. Output shape per image:", encoder.output_shape)

sample_images = captions_df["image"].unique()[:5]
sample_paths = [os.path.join("../data/Images", img) for img in sample_images]
sample_features = extract_features(encoder, sample_paths)


for path, feat in sample_features.items():
    print(path, "→ feature shape:", feat.shape)

Encoder loaded. Output shape per image: (None, None, None, 1280)
../data/Images\1000268201_693b08cb0e.jpg → feature shape: (49, 1280)
../data/Images\1001773457_577c3a7d70.jpg → feature shape: (49, 1280)
../data/Images\1002674143_1b742ab4b8.jpg → feature shape: (49, 1280)
../data/Images\1003163366_44323f5815.jpg → feature shape: (49, 1280)
../data/Images\1007129816_e794419615.jpg → feature shape: (49, 1280)


In [10]:
import time

all_images = captions_df["image"].unique().tolist()
all_paths = [os.path.join("../data/Images", img) for img in all_images]

print(f"Extracting features for {len(all_paths)} images...")
start = time.time()

all_features = extract_features(encoder, all_paths, batch_size=32)

elapsed = time.time() - start
print(f"Done in {elapsed/60:.1f} minutes")

Extracting features for 8091 images...
Done in 3.6 minutes


In [11]:
import pickle

with open("../data/image_features.pkl", "wb") as f:
    pickle.dump(all_features, f)

print("Features saved.")

Features saved.


In [14]:
sys.path.append("../src")
from decoder import CaptionDecoder
import numpy as np
import tensorflow as tf

vocab_size = len(word_to_idx)
embedding_dim = 256
units = 512

decoder = CaptionDecoder(vocab_size, embedding_dim, units)

dummy_features = tf.random.normal((1, 49, 1280))
hidden, cell = decoder.reset_state(batch_size=1)
dummy_input = tf.expand_dims([word_to_idx["startseq"]], 0)

output, hidden, cell, attn_weights = decoder(dummy_input, dummy_features, hidden, cell)

print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)

Output shape: (1, 2987)
Attention weights shape: (1, 49, 1)


In [15]:
import importlib
import data_loader
importlib.reload(data_loader)
from data_loader import captions_to_sequences

sequences = captions_to_sequences(captions_df["clean_caption"].tolist(), word_to_idx, max_len)
print("Sequences shape:", sequences.shape)
print("Sample sequence:", sequences[0])

image_list = captions_df["image"].tolist()
print("Number of image references:", len(image_list))

Sequences shape: (40455, 37)
Sample sequence: [2479    1  491 1274    1 1871  763 1307  525 2818    1 2216 1691 2464
 1274   49 2899  831    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0]
Number of image references: 40455


In [ ]:
import sys
sys.path.append("../src")
import importlib
import train
importlib.reload(train)
from train import train_model

small_df = captions_df.iloc[:500]
small_sequences = sequences[:500]

with open("../data/image_features.pkl", "rb") as f:
    image_features = pickle.load(f)

decoder, loss_history = train_model(
    small_df, small_sequences, image_features, word_to_idx,
    vocab_size=len(word_to_idx), batch_size=16, epochs=3
)